# Day Ahead Energy Price Forecast

In [52]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [53]:
import os

# Change the current working directory to the 'master_colab' folder on Google Drive
os.chdir('/content/drive/MyDrive/master_colab')

# Verify the current working directory
print(f"Current working directory: {os.getcwd()}")

Current working directory: /content/drive/MyDrive/master_colab


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import holidays
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import mutual_info_regression

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

import tensorflow as tf
from tensorflow.keras import layers, callbacks
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

import joblib
import pyarrow

# Load Data

In [5]:
# ENTSO-E day ahead price and load forecast data
df = pd.read_parquet("ENTSOE/entsoe_price_and_loadfc.parquet")

# for EDA change timezone to Berlin
df.index = df.index.tz_convert("Europe/Berlin")

# filter for 2023-2025
df = df.loc["2023":"2025"]

FileNotFoundError: [Errno 2] No such file or directory: 'ENTSOE/entsoe_price_and_loadfc.parquet'

In [ ]:
# Load previously saved generation forecast parquet files and merge into df
gen_fc_dir = Path("generation_forecast")

gen_fc_wind_on = pd.read_parquet(gen_fc_dir / "wind_on_generation_forecast.parquet").rename(
    columns={"generation_forecast": "gen_fc_wind_on"}
 )
gen_fc_wind_off = pd.read_parquet(gen_fc_dir / "wind_off_generation_forecast.parquet").rename(
    columns={"generation_forecast": "gen_fc_wind_off"}
 )
gen_fc_solar = pd.read_parquet(gen_fc_dir / "solar_generation_forecast.parquet").rename(
    columns={"generation_forecast": "gen_fc_solar"}
 )

# Combine and align timezone to df index
gen_fc = pd.concat([gen_fc_wind_on, gen_fc_wind_off, gen_fc_solar], axis=1).sort_index()
gen_fc = gen_fc.tz_convert("Europe/Berlin")

# Join to existing dataframe
df = df.join(gen_fc, how="left")


# Data Integrity & Time Index Quality

In [ ]:
# Ensure expected period
expected_index = pd.date_range(start="2023-01-01 00:00:00+00:00", end="2025-12-31 23:00:00+00:00", freq="h")
missing_timestamps = expected_index.difference(df.index)
extra_timestamps = df.index.difference(expected_index)

quality_checks = {
    "rows": len(df),
    "expected_rows": len(expected_index),
    "is_monotonic_increasing": bool(df.index.is_monotonic_increasing),
    "has_duplicate_index": bool(df.index.duplicated().any()),
    "missing_hours": int(len(missing_timestamps)),
    "extra_hours": int(len(extra_timestamps)),
    "timezone": str(df.index.tz),
}

quality_df = pd.DataFrame.from_dict(quality_checks, orient="index", columns=["value"])
quality_df

,value
rows,26304
expected_rows,26304
is_monotonic_increasing,True
has_duplicate_index,False
missing_hours,1
extra_hours,1
timezone,Europe/Berlin


In [ ]:
nan_count = df.isna().sum().rename("nan_count")
has_missing = df.isna().any().rename("has_missing")

quality_features = pd.concat([nan_count, has_missing], axis=1)
quality_features


,nan_count,has_missing
da_price,0,False
load_fc,0,False
gen_fc_wind_on,11688,True
gen_fc_wind_off,11688,True
gen_fc_solar,11688,True


First Figure: Some strong outliers that could have a too big impact on the model. However, these are not errors, so these extremes should also be described by the model.

Second Figure: Day ahead price is almost normally distributed but slightly left skewed.

Third Figure: Yearly seasonality visible. Probably month information important for model.

Forth Figure: Strong daily pattern. "Hour of day" variable useful for the model.

Fifth Figure: Strong weekly pattern. "Day of the week" variable useful for the model.

# Add features

# ____________________ FÜR GOOGLE COLAB ____________________

In [2]:
data = pd.read_parquet("entsoe_price_and_loadfc.parquet")
gen_fc_wind_on = pd.read_parquet("wind_on_generation_forecast.parquet").rename(
    columns={"generation_forecast": "gen_fc_wind_on"}
 )
gen_fc_wind_off = pd.read_parquet("wind_off_generation_forecast.parquet").rename(
    columns={"generation_forecast": "gen_fc_wind_off"}
 )
gen_fc_solar = pd.read_parquet("solar_generation_forecast.parquet").rename(
    columns={"generation_forecast": "gen_fc_solar"}
 )

gen_fc = pd.concat([gen_fc_wind_on, gen_fc_wind_off, gen_fc_solar], axis=1).sort_index()

data = data.join(gen_fc, how="left")

# ____________________ FÜR GOOGLE COLAB ____________________

In [3]:
# lag features

## convert to local time to handle daylight saving time correctly
data = data.tz_convert("Europe/Berlin")

idx_naive = data.index.tz_localize(None)

prev_naive_day = idx_naive - pd.DateOffset(days = 1)
prev_naive_day2 = idx_naive - pd.DateOffset(days = 2)
prev_naive_week = idx_naive - pd.DateOffset(weeks = 1)

idx_prev_day = prev_naive_day.tz_localize(
    "Europe/Berlin",
    ambiguous= False,
    nonexistent="shift_forward",
)
idx_prev_day2 = prev_naive_day2.tz_localize(
    "Europe/Berlin",
    ambiguous= False,
    nonexistent="shift_forward",
)
idx_prev_week = prev_naive_week.tz_localize(
    "Europe/Berlin",
    ambiguous= False,
    nonexistent="shift_forward",
)


data["price_lag_24h"] = data["da_price"].reindex(idx_prev_day).to_numpy()
data["price_lag_48h"] = data["da_price"].reindex(idx_prev_day2).to_numpy()
data["price_lag_168h"] = data["da_price"].reindex(idx_prev_week).to_numpy()


# weekday
data["weekday_sin"] = np.sin(2 * np.pi * data.index.dayofweek / 7)  # cyclic, so that distances are correct (Sunday is close to Monday), same for hour
data["weekday_cos"] = np.cos(2 * np.pi * data.index.dayofweek / 7)

# hour of day
data["hour_sin"] = np.sin(2 * np.pi * data.index.hour / 24)
data["hour_cos"] = np.cos(2 * np.pi * data.index.hour / 24)

# holiday not sunday
ger_holidays = list(holidays.Germany(years=data.index.year.unique()))
ger_holidays = pd.to_datetime(ger_holidays).tz_localize(data.index.tz)
data["holiday_not_sunday"] = (data.index.normalize().isin(ger_holidays) & (data.index.dayofweek != 6))


## back to utc
data = data.tz_convert("UTC")

In [4]:
# remove all data outside of 2023-2025 (only used for lag features)
data = data["2023":"2025"]

In [5]:
feature_cols = [
    "load_fc",
    "gen_fc_wind_on",
    "gen_fc_wind_off",
    "gen_fc_solar",
    "price_lag_24h",
    "price_lag_48h",
    "price_lag_168h",
    "weekday_sin",
    "weekday_cos",
    "hour_sin",
    "hour_cos",
    "holiday_not_sunday",
]

feature_eval = pd.DataFrame(index=feature_cols)
feature_eval["missing_ratio"] = data[feature_cols].isna().mean()
feature_eval["corr_with_price"] = [data["da_price"].corr(data[c]) for c in feature_cols]

clean_for_mi = data[["da_price"] + feature_cols].copy()
X_mi = clean_for_mi[feature_cols].copy()
X_mi["holiday_not_sunday"] = X_mi["holiday_not_sunday"].astype(int)
mi = mutual_info_regression(X_mi, clean_for_mi["da_price"], random_state=42)
feature_eval["mutual_info"] = mi

feature_eval.loc["hour_combined", "missing_ratio"] = feature_eval.loc[["hour_sin", "hour_cos"], "missing_ratio"].max()
feature_eval.loc["hour_combined", "corr_with_price"] = np.sqrt(
    feature_eval.loc["hour_sin", "corr_with_price"] ** 2 + feature_eval.loc["hour_cos", "corr_with_price"] ** 2
)
feature_eval.loc["hour_combined", "mutual_info"] = np.nan

feature_eval.loc["weekday_combined", "missing_ratio"] = feature_eval.loc[["weekday_sin", "weekday_cos"], "missing_ratio"].max()
feature_eval.loc["weekday_combined", "corr_with_price"] = np.sqrt(
    feature_eval.loc["weekday_sin", "corr_with_price"] ** 2 + feature_eval.loc["weekday_cos", "corr_with_price"] ** 2
)
feature_eval.loc["weekday_combined", "mutual_info"] = np.nan

for y in [2023, 2024, 2025]:
    y_mask = data.index.year == y
    year_corr = pd.Series({
        c: data.loc[y_mask, "da_price"].corr(data.loc[y_mask, c]) for c in feature_cols
    })
    feature_eval[f"corr_{y}"] = year_corr
    feature_eval.loc["hour_combined", f"corr_{y}"] = np.sqrt(
        feature_eval.loc["hour_sin", f"corr_{y}"] ** 2 + feature_eval.loc["hour_cos", f"corr_{y}"] ** 2
    )
    feature_eval.loc["weekday_combined", f"corr_{y}"] = np.sqrt(
        feature_eval.loc["weekday_sin", f"corr_{y}"] ** 2 + feature_eval.loc["weekday_cos", f"corr_{y}"] ** 2
    )


corr_cols = ["corr_2023", "corr_2024", "corr_2025"]
feature_eval["corr_std_across_years"] = feature_eval[corr_cols].std(axis=1)
feature_eval


,missing_ratio,corr_with_price,mutual_info,corr_2023,corr_2024,corr_2025,corr_std_across_years
load_fc,0.0000,0.2841,0.1229,0.3633,0.2335,0.2686,0.0671
gen_fc_wind_on,0.0000,-0.4107,0.1539,-0.4883,-0.3827,-0.3998,0.0567
gen_fc_wind_off,0.0000,-0.2920,0.0879,-0.3445,-0.3095,-0.2223,0.0630
gen_fc_solar,0.0000,-0.2239,0.0971,-0.1422,-0.1756,-0.3438,0.1080
price_lag_24h,0.0000,0.6699,0.4256,0.6418,0.6475,0.6988,0.0314
price_lag_48h,0.0000,0.4872,0.2399,0.4566,0.4281,0.5464,0.0617
price_lag_168h,0.0000,0.5075,0.2718,0.4832,0.4562,0.5518,0.0493
weekday_sin,0.0000,0.2087,0.0443,0.2240,0.2217,0.1885,0.0199
weekday_cos,0.0000,-0.0557,0.0101,-0.0687,-0.0664,-0.0337,0.0196
hour_sin,0.0000,-0.1069,0.0698,-0.1152,-0.1171,-0.0920,0.0140


In [6]:
# Initial feature whitelist based on low missingness and stable yearly behavior
feature_whitelist = feature_eval[(feature_eval["missing_ratio"] <= 0.02) & (feature_eval["corr_std_across_years"] <= 0.20) & (feature_eval["mutual_info"].notna())].index.tolist()

feature_whitelist

['load_fc',
 'gen_fc_wind_on',
 'gen_fc_wind_off',
 'gen_fc_solar',
 'price_lag_24h',
 'price_lag_48h',
 'price_lag_168h',
 'weekday_sin',
 'weekday_cos',
 'hour_sin',
 'hour_cos',
 'holiday_not_sunday']

# Split data

In [7]:
TARGET = "da_price"
TRAIN_START = pd.Timestamp("2023-01-01", tz="UTC")
TRAIN_END = pd.Timestamp("2025-01-01", tz="UTC")
TEST_START = pd.Timestamp("2025-01-01", tz="UTC")
TEST_END = pd.Timestamp("2026-01-01", tz="UTC")

train_mask = (data.index >= TRAIN_START) & (data.index < TRAIN_END)
test_mask = (data.index >= TEST_START) & (data.index < TEST_END)

data_train = data.loc[train_mask].copy()
data_test = data.loc[test_mask].copy()


split_summary = pd.DataFrame({
    "split": ["train", "test"],
    "start": [data_train.index.min(), data_test.index.min()],
    "end": [data_train.index.max(), data_test.index.max()],
    "rows": [len(data_train), len(data_test)],
})
split_summary

,split,start,end,rows
0,train,2023-01-01 00:00:00+00:00,2024-12-31 23:00:00+00:00,17544
1,test,2025-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,8760


# Baseline model

### Error Functions

In [8]:
def mae_score(y_true, y_pred) -> float:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse_score(y_true, y_pred) -> float:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

In [17]:
y_test = data_test[TARGET]
pred_daily = data_test["price_lag_24h"]
pred_weekly = data_test["price_lag_168h"]

baseline_scores = pd.DataFrame(
    [
        {"baseline": "daily_persistence_lag24", "mae": mae_score(y_test, pred_daily), "rmse": rmse_score(y_test, pred_daily)},
        {"baseline": "weekly_persistence_lag168", "mae": mae_score(y_test, pred_weekly), "rmse": rmse_score(y_test, pred_weekly)},
    ]
).sort_values("mae")

baseline_scores

,baseline,mae,rmse
0,daily_persistence_lag24,25.9650,40.4489
1,weekly_persistence_lag168,32.8043,49.3715


# LSTM-Readiness Checks

### General Setup

In [9]:
# Compact setup vars
LOOKBACK = 168
HORIZON_STEPS = 1
TARGET_COL = TARGET

FEATURE_COLS = ['load_fc',
                'gen_fc_wind_on',
                'gen_fc_wind_off',
                'gen_fc_solar',
                'price_lag_24h',
                'price_lag_48h',
                'price_lag_168h',
                'weekday_sin',
                'weekday_cos',
                'hour_sin',
                'hour_cos',
                'holiday_not_sunday']
#[c for c in feature_whitelist if c in data.columns]
SCALE_COLS = [
    c for c in FEATURE_COLS
    if c in ["load_fc", "gen_fc_wind_on", "gen_fc_wind_off", "gen_fc_solar", "price_lag_24h", "price_lag_48h", "price_lag_168h"]
]

model_df = data[[TARGET_COL] + FEATURE_COLS].copy()
train_df = model_df.loc[(model_df.index >= TRAIN_START) & (model_df.index < TRAIN_END)].copy()
test_df = model_df.loc[(model_df.index >= TEST_START) & (model_df.index < TEST_END)].copy()
test_history_df = model_df.loc[
    (model_df.index >= TEST_START - pd.Timedelta(hours=LOOKBACK)) & (model_df.index < TEST_START)
].copy()
test_with_history_df = pd.concat([test_history_df, test_df], axis=0)

# helper functions for target transformation (for robustness against outliers)
c = np.median(np.abs(train_df[TARGET_COL]))
def asinh_transform(y, c):
    return np.arcsinh(y / c)
def asinh_inverse(y_t, c):
    return c * np.sinh(y_t)
# raw truth
y_test = test_df[TARGET_COL].to_numpy(dtype=np.float32)


scaler = RobustScaler()
train_df.loc[:, SCALE_COLS] = scaler.fit_transform(train_df[SCALE_COLS])
train_df.loc[:, TARGET_COL] = asinh_transform(train_df[TARGET_COL], c)
test_df.loc[:, SCALE_COLS] = scaler.transform(test_df[SCALE_COLS])
test_df.loc[:, TARGET_COL] = asinh_transform(test_df[TARGET_COL], c)
test_history_df.loc[:, TARGET_COL] = asinh_transform(test_history_df[TARGET_COL].to_numpy(np.float32), c)
test_with_history_df = pd.concat([test_history_df, test_df], axis=0)
test_with_history_df.loc[test_df.index, SCALE_COLS] = test_df[SCALE_COLS]

### No-Lookback Setup

In [10]:
def build_point_dataset(df: pd.DataFrame, feature_cols: list[str], target_col: str):
    X = df[feature_cols].to_numpy(dtype=np.float32)
    y = df[target_col].to_numpy(dtype=np.float32)
    ts = df.index
    return X, y, ts


X_train_point, y_train_point, ts_train_point = build_point_dataset(train_df, FEATURE_COLS, TARGET_COL)
X_test_point, y_test_point, ts_test_point = build_point_dataset(test_df, FEATURE_COLS, TARGET_COL)

pd.DataFrame([
    {"set": "point", "split": "train", "X": X_train_point.shape, "y": y_train_point.shape},
    {"set": "point", "split": "test", "X": X_test_point.shape, "y": y_test_point.shape},
])

,set,split,X,y
0,point,train,"(17544, 12)","(17544,)"
1,point,test,"(8760, 12)","(8760,)"


### Lookback Setup

In [11]:
# Lookback setup: fixed 168h window ending at D-1 23:00 for every target timestamp.
# All hours of the same day share the same price sequence.
SEQUENCE_COLS = [c for c in [TARGET_COL] if c in model_df.columns]


def build_lookback_dataset(
    df: pd.DataFrame,
    sequence_cols: list[str],
    exog_cols: list[str],
    target_col: str,
    lookback_hours: int,
):
    """
    For each target timestamp (day D, hour h) the lookback window covers the
    168 hours ending at D-1 23:00.  All 24 hours of the same day therefore
    receive the identical price sequence; only the exog features differ.
    """
    idx = df.index
    sequence_values = df[sequence_cols].to_numpy(dtype=np.float32)
    exog_values     = df[exog_cols].to_numpy(dtype=np.float32)
    target_values   = df[target_col].to_numpy(dtype=np.float32)
    hours           = np.array([ts.hour for ts in idx], dtype=np.int32)

    X_seq, X_exog, y_target, ts_target = [], [], [], []

    for row_idx in range(lookback_hours, len(df)):
        h        = hours[row_idx]
        # D-1 23:00 is (h+1) hours before ts  →  row_idx - (h+1) in the index
        keep_idx = np.arange(row_idx - h - lookback_hours, row_idx - h)
        if keep_idx[0] < 0:
            continue
        X_seq.append(sequence_values[keep_idx, :])
        X_exog.append(exog_values[row_idx])
        y_target.append(target_values[row_idx])
        ts_target.append(idx[row_idx])

    return (
        np.stack(X_seq),
        np.stack(X_exog),
        np.asarray(y_target, dtype=np.float32),
        pd.DatetimeIndex(ts_target),
    )


# --- build datasets ---
X_train_seq_lookback, X_train_exog_lookback, y_train_lookback, ts_train_lookback = build_lookback_dataset(
    train_df, SEQUENCE_COLS, FEATURE_COLS, TARGET_COL, LOOKBACK,
)

X_test_seq_lookback, X_test_exog_lookback, y_test_lookback, ts_test_lookback = build_lookback_dataset(
    test_with_history_df, SEQUENCE_COLS, FEATURE_COLS, TARGET_COL, LOOKBACK,
)
test_keep_mask       = ts_test_lookback >= TEST_START
X_test_seq_lookback  = X_test_seq_lookback[test_keep_mask]
X_test_exog_lookback = X_test_exog_lookback[test_keep_mask]
y_test_lookback      = y_test_lookback[test_keep_mask]
ts_test_lookback     = ts_test_lookback[test_keep_mask]

# flat layout for XGBoost
X_train_flat_lookback = np.concatenate([
    X_train_seq_lookback.reshape(X_train_seq_lookback.shape[0], -1),
    X_train_exog_lookback,
], axis=1)
X_test_flat_lookback = np.concatenate([
    X_test_seq_lookback.reshape(X_test_seq_lookback.shape[0], -1),
    X_test_exog_lookback,
], axis=1)

display(pd.DataFrame([
    {"split": "train", "X_seq": X_train_seq_lookback.shape, "X_exog": X_train_exog_lookback.shape, "X_flat": X_train_flat_lookback.shape, "y": y_train_lookback.shape},
    {"split": "test",  "X_seq": X_test_seq_lookback.shape,  "X_exog": X_test_exog_lookback.shape,  "X_flat": X_test_flat_lookback.shape,  "y": y_test_lookback.shape},
]))

,split,X_seq,X_exog,X_flat,y
0,train,"(17376, 168, 1)","(17376, 12)","(17376, 180)","(17376,)"
1,test,"(8760, 168, 1)","(8760, 12)","(8760, 180)","(8760,)"


## ____________________ XGBoost ____________________




In [12]:
xgb_base = dict(
    objective="reg:squarederror",
    random_state=1904,
    n_jobs=-1,
    device="cuda"
)

param_dist = {
    "n_estimators": [100, 200, 400, 500],
    "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08],
    "max_depth": [3, 4, 5, 6, 8],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_lambda": [0.5, 1.0, 2.0, 5.0],
    "min_child_weight": [1, 3, 5, 7],
}

In [67]:
# Point-Model tuning
tscv_point = TimeSeriesSplit(n_splits=5)

search_point = RandomizedSearchCV(
    estimator=XGBRegressor(**xgb_base),
    param_distributions=param_dist,
    n_iter=30,
    scoring="neg_mean_absolute_error",  # oder neg_root_mean_squared_error
    cv=tscv_point,
    verbose=1,
    random_state=1904,
    n_jobs=1,
)

search_point.fit(X_train_point, y_train_point)
xgb_point_model = search_point.best_estimator_
y_pred_xgb_point = asinh_inverse(xgb_point_model.predict(X_test_point), c)

print("Best params (point):", search_point.best_params_)
print("Best CV score (point):", -search_point.best_score_)


Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best params (point): {'subsample': 0.6, 'reg_lambda': 0.5, 'n_estimators': 500, 'min_child_weight': 3, 'max_depth': 6, 'learning_rate': 0.01, 'colsample_bytree': 0.6}
Best CV score (point): 0.13820775747299194


In [68]:
# Lookback-Model tuning
tscv_lookback = TimeSeriesSplit(n_splits=5)

search_lookback = RandomizedSearchCV(
    estimator=XGBRegressor(**xgb_base),
    param_distributions=param_dist,
    n_iter=30,
    scoring="neg_mean_absolute_error",
    cv=tscv_lookback,
    verbose=1,
    random_state=1904,
    n_jobs=1,
)

search_lookback.fit(X_train_flat_lookback, y_train_lookback)
xgb_lookback_model = search_lookback.best_estimator_

y_pred_xgb_lookback = asinh_inverse(
    xgb_lookback_model.predict(X_test_flat_lookback).astype(np.float32), c
)

print("Best params (lookback):", search_lookback.best_params_)
print("Best CV score (lookback):", -search_lookback.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best params (lookback): {'subsample': 0.6, 'reg_lambda': 2.0, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.03, 'colsample_bytree': 1.0}
Best CV score (lookback): 0.1344897821545601


In [80]:
# save models
joblib.dump(xgb_point_model, "xgb_point_model.joblib")
joblib.dump(xgb_lookback_model, "xgb_lookback_model.joblib")


['xgb_lookback_model.joblib']

In [13]:
# load models
xgb_point = joblib.load("xgb_point_model.joblib")
xgb_lookback = joblib.load("xgb_lookback_model.joblib")

/opt/anaconda3/envs/entsoe/lib/python3.11/pickle.py:1718: UserWarning: [10:59:07] WARNING: /Users/runner/work/xgboost/xgboost/src/gbm/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


In [14]:
y_pred_xgb_point_t = xgb_point.predict(X_test_point)
y_pred_xgb_point   = asinh_inverse(y_pred_xgb_point_t, c)

y_pred_xgb_lookback_t = xgb_lookback.predict(X_test_flat_lookback)
y_pred_xgb_lookback   = asinh_inverse(y_pred_xgb_lookback_t, c)

pd.DataFrame([
    {"model": "xgb_point",    "mae": mae_score(y_test, y_pred_xgb_point),    "rmse": rmse_score(y_test, y_pred_xgb_point)},
    {"model": "xgb_lookback", "mae": mae_score(y_test, y_pred_xgb_lookback), "rmse": rmse_score(y_test, y_pred_xgb_lookback)},
]).sort_values("mae")

,model,mae,rmse
1,xgb_lookback,15.3231,24.4240
0,xgb_point,15.9846,25.1600


## ____________________ LSTM ____________________





In [87]:
input_seq  = layers.Input(shape=(LOOKBACK, X_train_seq_lookback.shape[2]))
input_exog = layers.Input(shape=(X_train_exog_lookback.shape[1],))

hidden = layers.LSTM(64)(input_seq)
hidden = layers.Concatenate()([hidden, input_exog])
hidden = layers.Dense(64, activation="relu")(hidden)
hidden = layers.Dropout(0.2)(hidden)
output = layers.Dense(1)(hidden)

lstm_model = tf.keras.Model([input_seq, input_exog], output)
lstm_model.compile(optimizer="adam", loss="mae", metrics=[tf.keras.metrics.RootMeanSquaredError(name="rmse")])

early_stop = callbacks.EarlyStopping(patience=6, restore_best_weights=True)
_ = lstm_model.fit(
    [X_train_seq_lookback, X_train_exog_lookback],
    y_train_lookback,
    validation_split=0.1,
    epochs=40,
    batch_size=64,
    shuffle=False,
    callbacks=[early_stop],
    verbose=1,
)

y_pred_lstm_t = lstm_model.predict(
    [X_test_seq_lookback, X_test_exog_lookback], verbose=0
).reshape(-1)
y_pred_lstm = asinh_inverse(y_pred_lstm_t, c)

Epoch 1/40
245/245 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - loss: 0.2824 - rmse: 0.4137 - val_loss: 0.1648 - val_rmse: 0.2353
Epoch 2/40
245/245 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1790 - rmse: 0.2373 - val_loss: 0.1628 - val_rmse: 0.2222
Epoch 3/40
245/245 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1673 - rmse: 0.2217 - val_loss: 0.1567 - val_rmse: 0.2162
Epoch 4/40
245/245 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1602 - rmse: 0.2119 - val_loss: 0.1626 - val_rmse: 0.2234
Epoch 5/40
245/245 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.1557 - rmse: 0.2073 - val_loss: 0.1699 - val_rmse: 0.2334
Epoch 6/40
245/245 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.1505 - rmse: 0.2016 - val_loss: 0.1643 - val_rmse: 0.2303
Epoch 7/40
245/245 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1469 - rmse: 0.1965 - val_loss: 0.1688 - val_rmse: 0.2321
Epoch 8/40
245/245 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1481 - rmse: 0.1965 - val_loss: 0.1736 - val_rmse: 0.2386
Epoch 9/40
245/245 ━━━━━━━━━━━━━━━━━━

In [ ]:
#load lstm model
lstm_model = joblib.load("lstm_model.joblib")

y_pred_lstm_t = lstm_model.predict(
    [X_test_seq_lookback, X_test_exog_lookback], verbose=200
).reshape(-1)
y_pred_lstm = asinh_inverse(y_pred_lstm_t, c)

In [ ]:
pd.DataFrame([
    {"model": "xgb_lookback",  "mae": mae_score(y_test, y_pred_xgb_lookback), "rmse": rmse_score(y_test, y_pred_xgb_lookback)},
    {"model": "lstm_lookback", "mae": mae_score(y_test, y_pred_lstm),           "rmse": rmse_score(y_test, y_pred_lstm)},
    {"model": "xgb_point",     "mae": mae_score(y_test, y_pred_xgb_point),     "rmse": rmse_score(y_test, y_pred_xgb_point)},
    {"model": "baseline_lag24","mae": mae_score(y_test, pred_daily),                "rmse": rmse_score(y_test, pred_daily)},
]).sort_values("mae")

## ____________________ SARIMAX ____________________





In [45]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

#SARIMAX_ORDER    = (1, 0, 0)
#SARIMAX_SEASONAL = (1, 1, 0, 24)
SARIMAX_ORDER    = (2, 0, 2)
SARIMAX_SEASONAL = (2, 1, 1, 24)
SARIMAX_EXOG_COLS = [
    "load_fc", "gen_fc_wind_on", "gen_fc_wind_off", "gen_fc_solar",
    "weekday_sin", "weekday_cos", "hour_sin", "hour_cos", "holiday_not_sunday",
]

In [46]:
sarimax_endog_train = train_df[TARGET_COL].astype(np.float64)
sarimax_exog_train  = train_df[SARIMAX_EXOG_COLS].astype(np.float64).ffill().bfill()

print(f"Starting SARIMAX fit | endog={sarimax_endog_train.shape} | exog={sarimax_exog_train.shape}")

sarimax_iter = {"i": 0}
def sarimax_callback(params):
    sarimax_iter["i"] += 1
    if sarimax_iter["i"] == 1 or sarimax_iter["i"] % 5 == 0:
        print(f"SARIMAX iteration {sarimax_iter['i']}")

sarimax_spec = SARIMAX(
    endog=sarimax_endog_train,
    exog=sarimax_exog_train,
    order=SARIMAX_ORDER,
    seasonal_order=SARIMAX_SEASONAL,
    trend="n",
    enforce_stationarity=False,
    enforce_invertibility=False,
    freq="h",
)
sarimax_fit = sarimax_spec.fit(
    disp=2,
    maxiter=200,
    method="lbfgs",
    callback=sarimax_callback,
)

print("\nSARIMAX fit finished.")
print("Optimizer return values:")
print(sarimax_fit.mle_retvals)
print(sarimax_fit.summary())

Starting SARIMAX fit | endog=(17544,) | exog=(17544, 9)


/opt/anaconda3/envs/entsoe/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency h will be used.
  self._init_dates(dates, freq)


SARIMAX iteration 1
SARIMAX iteration 5
SARIMAX iteration 10
SARIMAX iteration 15
SARIMAX iteration 20
SARIMAX iteration 25
SARIMAX iteration 30
SARIMAX iteration 35
SARIMAX iteration 40
SARIMAX iteration 45
SARIMAX iteration 50
SARIMAX iteration 55
SARIMAX iteration 60
SARIMAX iteration 65
SARIMAX iteration 70
SARIMAX iteration 75
SARIMAX iteration 80
SARIMAX iteration 85
SARIMAX iteration 90
SARIMAX iteration 95
SARIMAX iteration 100
SARIMAX iteration 105
SARIMAX iteration 110
SARIMAX iteration 115
SARIMAX iteration 120
SARIMAX iteration 125
SARIMAX iteration 130
SARIMAX iteration 135
SARIMAX iteration 140
SARIMAX iteration 145
SARIMAX iteration 150
SARIMAX iteration 155
SARIMAX iteration 160
SARIMAX iteration 165
SARIMAX iteration 170
SARIMAX iteration 175
SARIMAX iteration 180
SARIMAX iteration 185
SARIMAX iteration 190
SARIMAX iteration 195
SARIMAX iteration 200


/opt/anaconda3/envs/entsoe/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



SARIMAX fit finished.
Optimizer return values:
{'fopt': np.float64(-1.0915573568262575), 'gopt': array([ 0.00024298,  0.00065106,  0.00102563,  0.00400868, -0.00071742,
        0.00086577,  0.00094254,  0.00182966, -0.00225489, -0.04429542,
       -0.04167624, -0.00526201, -0.00421033, -0.00155453, -0.00159649,
       -0.00329433, -0.00647722]), 'fcalls': 4284, 'warnflag': 1, 'converged': False, 'iterations': 200}
                                      SARIMAX Results                                       
Dep. Variable:                             da_price   No. Observations:                17544
Model:             SARIMAX(2, 0, 2)x(2, 1, [1], 24)   Log Likelihood               19150.282
Date:                              Thu, 02 Apr 2026   AIC                         -38266.565
Time:                                      22:53:14   BIC                         -38134.504
Sample:                                  01-01-2023   HQIC                        -38223.074
                       

In [47]:
sarimax_endog_test = test_df[TARGET_COL].astype(np.float64)
sarimax_exog_test  = test_df[SARIMAX_EXOG_COLS].astype(np.float64).ffill().bfill()

test_dates = pd.date_range(
    start=pd.Timestamp(TEST_START).normalize(),
    end=pd.Timestamp(TEST_END) - pd.Timedelta(days=1),
    freq="D", tz="UTC",
)

sarimax_preds_t = {}
running_res = sarimax_fit  # startet vom Ende der Trainingsdaten
n_days = len(test_dates)

for i, day in enumerate(test_dates):
    exog_day = (
        sarimax_exog_test.loc[day : day + pd.Timedelta(hours=23)]
        .sort_index()
        .asfreq("h")
    )

    if len(exog_day) != 24:
        print(f"WARNING: {day.date()} has {len(exog_day)} hours, skipping.")
        continue

    # 24-Schritt-Forecast ab aktuellem Kalman-State (verwendet eigene Predictions als AR-Lags)
    forecast = running_res.get_forecast(steps=24, exog=exog_day)
    for ts, val in forecast.predicted_mean.items():
        sarimax_preds_t[ts] = val

    actual_day = (
        sarimax_endog_test.loc[day : day + pd.Timedelta(hours=23)]
        .sort_index()
        .asfreq("h")
    )
    running_res = running_res.extend(endog=actual_day, exog=exog_day)

    if (i + 1) % 10 == 0 or (i + 1) == n_days:
        print(f"  {i + 1}/{n_days} days done ({day.date()})")

print(f"Done: {len(sarimax_preds_t)} hourly predictions.")

  10/365 days done (2025-01-10)
  20/365 days done (2025-01-20)
  30/365 days done (2025-01-30)
  40/365 days done (2025-02-09)
  50/365 days done (2025-02-19)
  60/365 days done (2025-03-01)
  70/365 days done (2025-03-11)
  80/365 days done (2025-03-21)
  90/365 days done (2025-03-31)
  100/365 days done (2025-04-10)
  110/365 days done (2025-04-20)
  120/365 days done (2025-04-30)
  130/365 days done (2025-05-10)
  140/365 days done (2025-05-20)
  150/365 days done (2025-05-30)
  160/365 days done (2025-06-09)
  170/365 days done (2025-06-19)
  180/365 days done (2025-06-29)
  190/365 days done (2025-07-09)
  200/365 days done (2025-07-19)
  210/365 days done (2025-07-29)
  220/365 days done (2025-08-08)
  230/365 days done (2025-08-18)
  240/365 days done (2025-08-28)
  250/365 days done (2025-09-07)
  260/365 days done (2025-09-17)
  270/365 days done (2025-09-27)
  280/365 days done (2025-10-07)
  290/365 days done (2025-10-17)
  300/365 days done (2025-10-27)
  310/365 days done

In [48]:
sarimax = asinh_inverse(pd.Series(sarimax_preds_t),c)

print(f"SARIMAX  MAE={mae_score(y_test, sarimax):.4f}  RMSE={rmse_score(y_test, sarimax):.4f}")

SARIMAX  MAE=15.5992  RMSE=24.3286


In [49]:
preds_df = pd.DataFrame({"sarimax_pred": sarimax})
preds_df.to_parquet("sarimax_predictions_2.parquet")


In [39]:
sarimax_4exog_cols = sarimax

In [50]:
print(f"SARIMAX  MAE={mae_score(y_test, sarimax_4exog_cols):.4f}  RMSE={rmse_score(y_test, sarimax_4exog_cols):.4f}")

SARIMAX  MAE=17.1058  RMSE=26.5361


In [89]:
scores = pd.DataFrame([
    {"model": "xgb_lookback",  "mae": mae_score(y_test, y_pred_xgb_lookback), "rmse": rmse_score(y_test, y_pred_xgb_lookback)},
    {"model": "lstm_lookback", "mae": mae_score(y_test, y_pred_lstm),           "rmse": rmse_score(y_test, y_pred_lstm)},
    {"model": "xgb_point",     "mae": mae_score(y_test, y_pred_xgb_point),     "rmse": rmse_score(y_test, y_pred_xgb_point)},
    {"model": "baseline_lag24","mae": mae_score(y_test, pred_daily),                "rmse": rmse_score(y_test, pred_daily)},
]).sort_values("mae")

scores

,model,mae,rmse
0,xgb_lookback,15.3231,24.4240
2,xgb_point,15.9846,25.1600
1,lstm_lookback,17.5491,26.6549
3,baseline_lag24,25.9650,40.4489


#_______________________________________________________________________________________


In [ ]:
import plotly.graph_objects as go

y_true = np.asarray(y_test).reshape(-1)
y_pred = np.asarray(y_pred_xgb_lookback2).reshape(-1)

n = min(len(y_true), len(y_pred), len(ts_test_lookback))
df_plot = pd.DataFrame(
    {"Ist": y_true[:n], "XGB Lookback": y_pred[:n]},
    index=ts_test_lookback[:n],
)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot["Ist"],          mode="lines", name="Ist"))
fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot["XGB Lookback"], mode="lines", name="XGB Lookback"))

fig.update_layout(
    title="Ist vs. XGB Lookback (interaktiv zoombar)",
    xaxis_title="Zeit",
    yaxis_title="Preis [EUR/MWh]",
    template="plotly_white",
    hovermode="x unified",
    height=700,
)
fig.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(buttons=[
        dict(count=1,  label="1d", step="day",   stepmode="backward"),
        dict(count=7,  label="7d", step="day",   stepmode="backward"),
        dict(count=1,  label="1m", step="month", stepmode="backward"),
        dict(step="all", label="Alles"),
    ]),
)
fig.show()